In [14]:
import pandas as pd

def _as_list(value):
    """把单个值统一转成列表，方便批量循环。"""
    if isinstance(value, (list, tuple, pd.Series, pd.Index)):
        return list(value)
    return [value]

def add_months(dt, months):
    """按 Excel EDATE 逻辑加月份：目标月没有对应日期时取月末。"""
    return pd.Timestamp(dt) + pd.DateOffset(months=months)
    
def pmt(rate, nper, pv):
    """返回等额本息每期还款额，对应 Excel: -PMT(rate, nper, pv)。"""
    if rate == 0:
        return pv / nper
    return pv * rate / (1 - (1 + rate) ** (-nper))
    
def xirr(cashflows, dates, low=-0.999999, high=10, tol=1e-10, max_iter=1000):
    """用二分法计算 XIRR，现金流需同时包含正负值。"""
    f_low = xnpv(low, cashflows, dates)
    f_high = xnpv(high, cashflows, dates)

    while f_low * f_high > 0 and high < 1_000:
        high *= 2
        f_high = xnpv(high, cashflows, dates)

    if f_low * f_high > 0:
        raise ValueError("XIRR 无法求解：请检查现金流是否同时包含正负值，或扩大 high。")

    for _ in range(max_iter):
        mid = (low + high) / 2
        f_mid = xnpv(mid, cashflows, dates)

        if abs(f_mid) < tol:
            return mid

        if f_low * f_mid < 0:
            high = mid
            f_high = f_mid
        else:
            low = mid
            f_low = f_mid

    return mid

def xnpv(rate, cashflows, dates):
    """按真实日期折现的净现值。"""
    d0 = pd.Timestamp(dates[0])
    return sum(
        cf / ((1 + rate) ** ((pd.Timestamp(d) - d0).days / 365))
        for cf, d in zip(cashflows, dates)
    )

def build_monthly_repay_dates(start_date, periods):
    """按起息日逐月生成还款日期，对齐上面 add_months/Excel EDATE 逻辑。"""
    start_date = pd.Timestamp(start_date)
    return [add_months(start_date, i) for i in range(1, periods + 1)]

def calc_repay_plan_by_target_xirr(
    principal,
    start_date,
    repay_dates,
    target_converted_annual_rate=0.24,
    weights=None,
):
    """
    根据本金、起息日、还款日期列表、目标年化，用折现系数闭式解计算还款计划金额。

    口径：先把目标年化转成对应 XIRR，再按真实累计计息天数折现：
        目标XIRR = (1 + 目标年化 / 12)^12 - 1
        折现系数 = 1 / ((1 + 目标XIRR)^(累计计息天数 / 365))

    参数：
    - principal: 本金，例如 10000
    - start_date: 0期起息日/放款日，例如 "2025-01-15"
    - repay_dates: 还款日期列表，例如 ["2025-02-15", "2025-03-15", ...]
    - target_converted_annual_rate: 目标 XIRR折算年化，默认 24%
    - weights: 每期还款权重，默认每期等额；如果传入 [1, 2, 3]，则每期金额按 1:2:3 分配

    返回：
    - summary: 汇总结果，含目标折现XIRR、折现系数合计、精确还款额等
    - schedule: 现金流明细，含累计计息天数、折现系数和每期还款金额
    """
    start_date = pd.Timestamp(start_date)
    repay_dates = [pd.Timestamp(d) for d in repay_dates]
    periods = len(repay_dates)

    if periods == 0:
        raise ValueError("repay_dates 不能为空。")
    if any(d <= start_date for d in repay_dates):
        raise ValueError("所有还款日期都必须晚于起息日。")

    if weights is None:
        weights = [1] * periods
    if len(weights) != periods:
        raise ValueError("weights 长度必须和 repay_dates 一致。")

    weights = pd.Series(weights, dtype="float64")
    if (weights <= 0).any():
        raise ValueError("weights 必须全部大于 0。")

    # 目标定义：converted_annual_rate = xirr_period_rate * 12
    # 且 xirr_period_rate = (1 + XIRR)^(1/12) - 1
    # 所以先由目标折算年化反推出需要用于真实日期折现的目标 XIRR。
    target_xirr_period_rate = target_converted_annual_rate / 12
    target_discount_xirr = (1 + target_xirr_period_rate) ** 12 - 1

    cumulative_days = [(d - start_date).days for d in repay_dates]
    discount_factors = [
        1 / ((1 + target_discount_xirr) ** (days / 365))
        for days in cumulative_days
    ]
    discount_factor_sum = sum(w * df for w, df in zip(weights, discount_factors))

    # 闭式解：令 -principal + base_amount * sum(weight_i * discount_factor_i) = 0。
    base_amount = principal / discount_factor_sum
    repay_amounts = [base_amount * w for w in weights]

    cashflow_dates = [start_date] + repay_dates
    cashflows = [-principal] + repay_amounts

    # 精确解已按目标 XIRR 折现求出，不再用二分法反算 XIRR。
    annual_xirr = target_discount_xirr
    xirr_period_rate = target_xirr_period_rate
    converted_annual_rate = target_converted_annual_rate
    diff = 0.0

    summary = pd.DataFrame(
        {
            "0期起息日": [start_date.date()],
            "本金": [principal],
            "期数": [periods],
            "年化费率": [target_converted_annual_rate],
            "月度费率": [target_xirr_period_rate],
            "折现系数合计": [discount_factor_sum],
            "基准还款额": [base_amount],
            "反算XIRR": [annual_xirr],
            "XIRR折算周期费率": [xirr_period_rate],
            "XIRR折算年化": [converted_annual_rate],
            "折算年化与目标差": [diff],
            "折算年化与目标差_bp": [diff * 10_000],
        }
    )

    schedule = pd.DataFrame(
        {
            "期次": range(0, periods + 1),
            "日期": [d.date() for d in cashflow_dates],
            "累计计息天数": [0] + cumulative_days,
            "折现系数": [1] + discount_factors,
            "现金流": cashflows,
            "还款金额": [0] + repay_amounts,
        }
    )

    return summary, schedule


def calc_xirr_metrics(cashflows, dates):
    """返回真实日期 XIRR、等效月度费率和折算年化费率。"""
    annual_xirr = xirr(cashflows, dates)
    period_rate = (1 + annual_xirr) ** (1 / 12) - 1
    converted_annual_rate = period_rate * 12
    return annual_xirr, period_rate, converted_annual_rate


def calc_discount_factor_summary(converted_annual_rate, start_date, repay_dates):
    """按折算年化反推目标 XIRR，并计算每期累计天数折现系数。"""
    target_xirr = (1 + converted_annual_rate / 12) ** 12 - 1
    cumulative_days = [(d - start_date).days for d in repay_dates]
    discount_factors = [
        1 / ((1 + target_xirr) ** (days / 365))
        for days in cumulative_days
    ]
    return target_xirr, cumulative_days, discount_factors, sum(discount_factors)


def calc_rate_compare_row(
    principal,
    start_date,
    periods,
    annual_rate,
    actual_rate,
    rate_label=None,
):
    """
    单组参数输出一行汇总：
    - 原履约：总期供、资方期供、我方现金流及对应 XIRR
    - 方案一：原资方本金 + 精确担保费
    - 方案二：整体/资方分别按折现系数闭式解得到合规期供
    """
    start_date = pd.Timestamp(start_date)
    repay_dates = build_monthly_repay_dates(start_date, periods)
    cashflow_dates = [start_date] + repay_dates

    annual_pmt_amount = pmt(annual_rate / 12, periods, principal)
    actual_pmt_amount = pmt(actual_rate / 12, periods, principal)
    original_guarantee_fee = annual_pmt_amount - actual_pmt_amount

    remaining_principal = principal
    funder_principal_amounts = []
    funder_interest_amounts = []
    for _ in repay_dates:
        interest_amount = remaining_principal * actual_rate / 12
        principal_amount = actual_pmt_amount - interest_amount
        funder_interest_amounts.append(interest_amount)
        funder_principal_amounts.append(principal_amount)
        remaining_principal -= principal_amount

    original_total_payments = [annual_pmt_amount] * periods
    original_funder_payments = [actual_pmt_amount] * periods
    original_our_cashflows = [
        principal_amount + original_guarantee_fee
        for principal_amount in funder_principal_amounts
    ]

    annual_xirr, annual_period_rate, annual_converted_rate = calc_xirr_metrics(
        [-principal] + original_total_payments,
        cashflow_dates,
    )
    actual_xirr, actual_period_rate, actual_converted_rate = calc_xirr_metrics(
        [-principal] + original_funder_payments,
        cashflow_dates,
    )
    our_xirr, our_period_rate, our_converted_rate = calc_xirr_metrics(
        [-principal] + original_our_cashflows,
        cashflow_dates,
    )

    total_discount_xirr, cumulative_days, total_discount_factors, total_discount_factor_sum = calc_discount_factor_summary(
        annual_rate,
        start_date,
        repay_dates,
    )
    funder_discount_xirr, _, funder_discount_factors, funder_discount_factor_sum = calc_discount_factor_summary(
        actual_rate,
        start_date,
        repay_dates,
    )

    guarantee_discount_annual_rate = annual_rate - actual_converted_rate
    guarantee_discount_xirr, _, guarantee_discount_factors, guarantee_discount_factor_sum = calc_discount_factor_summary(
        guarantee_discount_annual_rate,
        start_date,
        repay_dates,
    )

    # 方案二：整体和资方分别用本金 / 折现系数和，保证折现后本金为 principal。
    plan2_total_payment = principal / total_discount_factor_sum
    plan2_funder_payment = principal / funder_discount_factor_sum
    plan2_guarantee_fee = plan2_total_payment - plan2_funder_payment
    plan2_total = plan2_total_payment * periods
    plan2_funder_total = plan2_funder_payment * periods
    plan2_guarantee_total = plan2_guarantee_fee * periods

    # 方案一：保留原资方本金节奏，反推固定担保费。
    discounted_original_principal = sum(
        principal_amount * df
        for principal_amount, df in zip(funder_principal_amounts, guarantee_discount_factors)
    )
    plan1_guarantee_fee = (
        principal - discounted_original_principal
    ) / guarantee_discount_factor_sum
    plan1_our_cashflows = [
        principal_amount + plan1_guarantee_fee
        for principal_amount in funder_principal_amounts
    ]
    plan1_total_payments = [
        interest_amount + our_cashflow
        for interest_amount, our_cashflow in zip(funder_interest_amounts, plan1_our_cashflows)
    ]
    plan1_guarantee_total = plan1_guarantee_fee * periods
    plan1_our_total = sum(plan1_our_cashflows)
    plan1_total = sum(plan1_total_payments)

    annual_pmt_total = sum(original_total_payments)
    actual_pmt_total = sum(original_funder_payments)
    original_guarantee_total = original_guarantee_fee * periods
    original_our_total = sum(original_our_cashflows)

    return {
        "本金": principal,
        "起息日": start_date.date(),
        "期限": periods,
        "首期还款日": repay_dates[0].date(),
        "末期还款日": repay_dates[-1].date(),
        "计息总天数": (repay_dates[-1] - start_date).days,
        "利率组合": rate_label if rate_label is not None else f"{annual_rate:.2%}/{actual_rate:.2%}",
        "年利率": annual_rate,
        "资方利率": actual_rate,
        "年利率PMT还款值": annual_pmt_amount,
        "资方利率PMT还款值": actual_pmt_amount,
        "原每期担保费": original_guarantee_fee,
        "年利率PMT累计值": annual_pmt_total,
        "资方利率PMT累计值": actual_pmt_total,
        "原担保费累计值": original_guarantee_total,
        "原我方现金流累计值": original_our_total,
        "原总期供XIRR": annual_xirr,
        "原总期供周期费率": annual_period_rate,
        "原总期供年化费率": annual_converted_rate,
        "原资方XIRR": actual_xirr,
        "原资方周期费率": actual_period_rate,
        "原资方年化费率": actual_converted_rate,
        "原我方XIRR": our_xirr,
        "原我方周期费率": our_period_rate,
        "原我方年化费率": our_converted_rate,
        "整体折现XIRR": total_discount_xirr,
        "资方折现XIRR": funder_discount_xirr,
        "我方担保折现年化": guarantee_discount_annual_rate,
        "我方担保折现XIRR": guarantee_discount_xirr,
        "整体折现系数合计": total_discount_factor_sum,
        "资方折现系数合计": funder_discount_factor_sum,
        "我方折现系数合计": guarantee_discount_factor_sum,
        "方案2整体合规期供": plan2_total_payment,
        "方案2资方合规期供": plan2_funder_payment,
        "方案2我方担保费": plan2_guarantee_fee,
        "方案2整体合规累计值": plan2_total,
        "方案2资方合规累计值": plan2_funder_total,
        "方案2我方担保费累计值": plan2_guarantee_total,
        "方案2整体合规累计-原总期供累计": plan2_total - annual_pmt_total,
        "方案1每期担保费": plan1_guarantee_fee,
        "方案1担保费累计值": plan1_guarantee_total,
        "方案1我方现金流累计值": plan1_our_total,
        "方案1整体累计值": plan1_total,
        "方案1我方现金流累计值-原我方现金流累计值": plan1_our_total - original_our_total,
    }


def batch_calc_rate_compare(
    principal=10_000,
    start_dates="2025-02-15",
    periods=12,
    rate_pairs=None,
):
    """
    批量汇总输出，适合直接拿去做透视表。

    参数示例：
    rate_pairs = [
        {"年利率": 0.24, "资方利率": 0.06},
        {"年利率": 0.36, "资方利率": 0.09},
    ]
    start_dates = ["2025-02-15", "2025-03-15"]
    periods = [3, 6, 12]
    """
    if rate_pairs is None:
        rate_pairs = [{"年利率": 0.24, "资方利率": 0.06}]
    elif isinstance(rate_pairs, dict):
        rate_pairs = [rate_pairs]
    elif isinstance(rate_pairs, tuple) and len(rate_pairs) >= 2 and not isinstance(rate_pairs[0], (dict, list, tuple)):
        rate_pairs = [rate_pairs]

    rows = []
    for start_date in _as_list(start_dates):
        for period in _as_list(periods):
            for pair in rate_pairs:
                if isinstance(pair, dict):
                    annual_rate = pair["年利率"]
                    actual_rate = pair["资方利率"]
                    rate_label = pair.get("利率组合")
                else:
                    annual_rate, actual_rate = pair[:2]
                    rate_label = pair[2] if len(pair) > 2 else None

                rows.append(
                    calc_rate_compare_row(
                        principal=principal,
                        start_date=start_date,
                        periods=int(period),
                        annual_rate=float(annual_rate),
                        actual_rate=float(actual_rate),
                        rate_label=rate_label,
                    )
                )

    return pd.DataFrame(rows)

In [ ]:
# 示例：你只需要改这里的输入，就能得到透视表用的汇总 dataframe
rate_pairs = [
    {"年利率": 0.2400, "资方利率": 0.0600},
]

result_df = batch_calc_rate_compare(
    principal=10_000,
    # start_dates=["2025-01-15","2025-02-15",
    #             "2025-03-15","2025-04-15",
    #             "2025-05-15","2025-06-15",
    #             "2025-07-15","2025-08-15",
    #             "2025-09-15","2025-10-15",
    #             "2025-11-15","2025-12-15",],
    start_dates = pd.date_range("2025-01-01", "2025-12-31", freq="D"),
    periods=[12,6,3,1],
    rate_pairs=rate_pairs,
)

# result_df.to_excel("result.xlsx", index=False)
with pd.ExcelWriter("D:\\10.LTV月度更新\\息费测算\\新规计息对比老方法.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    result_df.to_excel(writer, sheet_name="by月", index=False)

In [13]:
# 示例1：单个起息日，直接查看主要函数输出
principal = 10_000
periods = 12
start_date = "2015-02-15"
annual_rate = 0.24
actual_rate = 0.06
repay_dates = build_monthly_repay_dates(start_date, periods)

# 1. 查看 calc_repay_plan_by_target_xirr：年利率口径
annual_summary, annual_schedule = calc_repay_plan_by_target_xirr(
    principal=principal,
    start_date=start_date,
    repay_dates=repay_dates,
    target_converted_annual_rate=annual_rate,
)
display(annual_summary)
display(annual_schedule)

# 2. 查看 calc_repay_plan_by_target_xirr：资方利率口径
actual_summary, actual_schedule = calc_repay_plan_by_target_xirr(
    principal=principal,
    start_date=start_date,
    repay_dates=repay_dates,
    target_converted_annual_rate=actual_rate,
)
display(actual_summary)
display(actual_schedule)

# 3. 查看 calc_rate_compare_row：完整单 case 汇总
compare_row = calc_rate_compare_row(
    principal=principal,
    start_date=start_date,
    periods=periods,
    annual_rate=annual_rate,
    actual_rate=actual_rate,
)
compare_df = pd.DataFrame([compare_row])
display(compare_df)
display(compare_df.T.rename(columns={0: "输出值"}))


,0期起息日,本金,期数,年化费率,月度费率,折现系数合计,基准还款额,反算XIRR,XIRR折算周期费率,XIRR折算年化,折算年化与目标差,折算年化与目标差_bp
0,2015-02-15,10000,12,0.24,0.02,10.585113,944.723069,0.268242,0.02,0.24,0.0,0.0


,期次,日期,累计计息天数,折现系数,现金流,还款金额
0,0,2015-02-15,0,1.000000,-10000.000000,0.000000
1,1,2015-03-15,28,0.981936,944.723069,944.723069
2,2,2015-04-15,59,0.962317,944.723069,944.723069
3,3,2015-05-15,89,0.943704,944.723069,944.723069
4,4,2015-06-15,120,0.924848,944.723069,944.723069
5,5,2015-07-15,150,0.906960,944.723069,944.723069
6,6,2015-08-15,181,0.888839,944.723069,944.723069
7,7,2015-09-15,212,0.871080,944.723069,944.723069
8,8,2015-10-15,242,0.854232,944.723069,944.723069
9,9,2015-11-15,273,0.837164,944.723069,944.723069


,0期起息日,本金,期数,年化费率,月度费率,折现系数合计,基准还款额,反算XIRR,XIRR折算周期费率,XIRR折算年化,折算年化与目标差,折算年化与目标差_bp
0,2015-02-15,10000,12,0.06,0.005,11.621573,860.468717,0.061678,0.005,0.06,0.0,0.0


,期次,日期,累计计息天数,折现系数,现金流,还款金额
0,0,2015-02-15,0,1.000000,-10000.000000,0.000000
1,1,2015-03-15,28,0.995419,860.468717,860.468717
2,2,2015-04-15,59,0.990372,860.468717,860.468717
3,3,2015-05-15,89,0.985512,860.468717,860.468717
4,4,2015-06-15,120,0.980515,860.468717,860.468717
5,5,2015-07-15,150,0.975704,860.468717,860.468717
6,6,2015-08-15,181,0.970757,860.468717,860.468717
7,7,2015-09-15,212,0.965835,860.468717,860.468717
8,8,2015-10-15,242,0.961095,860.468717,860.468717
9,9,2015-11-15,273,0.956222,860.468717,860.468717


,本金,起息日,期限,首期还款日,末期还款日,计息总天数,利率组合,年利率,资方利率,年利率PMT还款值,...,方案2我方担保费,方案2整体合规累计值,方案2资方合规累计值,方案2我方担保费累计值,方案2整体合规累计-原总期供累计,方案1每期担保费,方案1担保费累计值,方案1我方现金流累计值,方案1整体累计值,方案1整体累计-原我方现金流累计值
0,10000,2015-02-15,12,2015-03-15,2016-02-15,365,24.00%/6.00%,0.24,0.06,945.595966,...,84.254352,11336.676827,10325.6246,1011.052227,-10.474767,83.372284,1000.467412,11000.467412,11328.438977,309.258947


,输出值
本金,10000
起息日,2015-02-15
期限,12
首期还款日,2015-03-15
末期还款日,2016-02-15
计息总天数,365
利率组合,24.00%/6.00%
年利率,0.24
资方利率,0.06
年利率PMT还款值,945.595966


In [ ]:
# 示例：你只需要改这里的输入，就能得到透视表用的汇总 dataframe
rate_pairs = [
    {"年利率": 0.2400, "资方利率": 0.0650},
    {"年利率": 0.2400, "资方利率": 0.0613},
    {"年利率": 0.2400, "资方利率": 0.0600},
    {"年利率": 0.2400, "资方利率": 0.0583},
    {"年利率": 0.2400, "资方利率": 0.0500},
    {"年利率": 0.2400, "资方利率": 0.0480},
    {"年利率": 0.2400, "资方利率": 0.0450},
    {"年利率": 0.2400, "资方利率": 0.0400},
    {"年利率": 0.2400, "资方利率": 0.0360},
]

result_df = batch_calc_rate_compare(
    principal=10_000,
    start_dates=["2025-02-15","2025-07-15",'2025-09-15'],
    periods=[12,6,3,1],
    rate_pairs=rate_pairs,
)

# result_df.to_excel("result.xlsx", index=False)
with pd.ExcelWriter("result.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    result_df.to_excel(writer, sheet_name="by月", index=False)

In [15]:
# 输出 2025 全年每日起息日、24%/6% 利率组合的数据
rate_pairs = [
    {"年利率": 0.2400, "资方利率": 0.0600},
]

result_df = batch_calc_rate_compare(
    principal=10_000,
    start_dates=pd.date_range("2025-01-01", "2025-12-31", freq="D"),
    periods=[12, 6, 3, 1],
    rate_pairs=rate_pairs,
)

with pd.ExcelWriter("D:\\10.LTV月度更新\\息费测算\\result.xlsx", engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    result_df.to_excel(writer, sheet_name="全年24_6", index=False)